# Sliding Window Debugging Notebook
- If **centralized is also infeasible** → bug is in state forwarding
- If **centralized is feasible but distributed is not** → bug is in ADMM local subproblem
- **SOC trajectories** across windows for both solvers
- **Cost per window** comparison

## 1. Setup

In [ ]:
import sys, json, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from pathlib import Path
REPO = Path('/Users/catharinahente/Documents/Github/p2p-energy-trading-illustrative')
sys.path.insert(0, str(REPO))

from scenarios import make_scenario
from algo      import (centralized, distributed,
                        centralized_window, distributed_window,
                        _slice_data)

plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.size':         9,
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

# ── Configuration ─────────────────────────────────────────────────────────────
SEASON      = 'winter'
SEED        = 42
RES         = 0.5       # 30-min resolution
WINDOW_H    = 3.0       # TO BE ADJUSTED
WITH_NET    = False     # True to include network constraints
RHO         = 2.0
TOL         = 1e-6
MAX_ITER    = 5000

data = make_scenario(SEASON, seed=SEED, resolution_h=RES)
N    = len(data['demand'])
T    = len(data['buy_price'])
dt   = data['dt']
Tw   = int(round(WINDOW_H / dt))
nw   = T // Tw
names = data['prosumer_names']

print(f'T={T}  dt={dt}h  Tw={Tw} steps/window  nw={nw} windows')
print(f'Window size: {WINDOW_H}h  Resolution: {int(RES*60)}min')

Building winter scenario (T=48, dt=0.5h, resolution=30min)...
  P1: Residential ++ (bus 5)
    HP: Air_Alternative_1 (Air alternative, Hanover)
  P2: Office building (bus 11)
    ⚠ No EV activity detected — using fallback schedule
  P3: Residential HP only (bus 17)
    HP: Air_Semi-Parallel_1 (Air semi-parallel, Hanover)
  P4: Residential PV only (bus 32)
  Done. N=4, T=48, dt=0.5h, max_demand=6.3 kW
T=48  dt=0.5h  Tw=6 steps/window  nw=8 windows
Window size: 3.0h  Resolution: 30min


## 2. Full-day reference (no windowing)

In [2]:
print('Running full-day centralized (reference)...')
result_full_c = centralized(data)
print(f'Full-day centralized: {result_full_c["costs"].sum():.4f} EUR')
for i in range(N):
    print(f'  P{i+1}: {result_full_c["costs"][i]:.4f} EUR  '
          f'SOC: {result_full_c["soc"][i,0]:.2f}→{result_full_c["soc"][i,-1]:.2f} kWh')

Running full-day centralized (reference)...
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2806500
Academic license 2806500 - for non-commercial use only - registered to ch___@ethz.ch
Full-day centralized: 58.6009 EUR
  P1: 7.4449 EUR  SOC: 4.50→4.50 kWh
  P2: 37.1328 EUR  SOC: 6.00→6.00 kWh
  P3: 3.2737 EUR  SOC: 0.00→0.00 kWh
  P4: 10.7494 EUR  SOC: 3.00→3.00 kWh


## 3. Window-by-window diagnostic
Runs centralized and distributed on each window with **identical** initial conditions.
Prints detailed state at each window boundary.

In [3]:
# Run centralized window
print(f'\n{"="*60}')
print(f'CENTRALIZED WINDOW  ({nw} × {WINDOW_H}h)')
print(f'{"="*60}')
t0 = time.time()
result_cent_w = centralized_window(data, window_h=WINDOW_H,
                                    with_network=WITH_NET)
print(f'\nCentralized window: {time.time()-t0:.1f}s')
if result_cent_w:
    print(f'Total cost: {result_cent_w["costs"].sum():.4f} EUR')
    print(f'vs full-day: {result_full_c["costs"].sum():.4f} EUR  '
          f'(Δ={result_cent_w["costs"].sum()-result_full_c["costs"].sum():+.4f})')


CENTRALIZED WINDOW  (8 × 3.0h)
Centralized sliding window: 8 × 3.0h (6 steps)
  Window 1/8  t=[0:6)  (free terminal)
    soc_now:    [4.5, 6.0, 0.0, 3.0]
    ev_soc_now: [43.2, 28.8, 0.0, 0.0]
    T_now:      [20.0, 20.0, 20.0, 20.0]
    ✓ cost=4.1797 EUR
  Window 2/8  t=[6:12)  (free terminal)
    soc_now:    [1.5, 2.0, 0.0, 1.0]
    ev_soc_now: [42.379, 27.525, 0.0, 0.0]
    T_now:      [20.0, 20.0, 20.0, 20.0]
    ✓ cost=6.1871 EUR
  Window 3/8  t=[12:18)  (free terminal)
    soc_now:    [1.5, 2.0, 0.0, 1.0]
    ev_soc_now: [41.559, 26.25, 0.0, 0.0]
    T_now:      [19.926, 20.0, 19.926, 20.0]
    ✓ cost=7.6571 EUR
  Window 4/8  t=[18:24)  (free terminal)
    soc_now:    [1.5, 2.0, 0.0, 1.0]
    ev_soc_now: [40.738, 22.4, 0.0, 0.0]
    T_now:      [19.879, 20.0, 19.879, 20.0]
    ✓ cost=4.7761 EUR
  Window 5/8  t=[24:30)  (free terminal)
    soc_now:    [1.5, 2.0, 0.0, 1.0]
    ev_soc_now: [39.917, 22.4, 0.0, 0.0]
    T_now:      [19.762, 20.0, 19.762, 20.0]
    ✓ cost=5.1326 EUR
 

In [ ]:
# Run distributed window — only if centralized succeeded
if result_cent_w is not None:
    print(f'\n{"="*60}')
    print(f'DISTRIBUTED WINDOW  ({nw} × {WINDOW_H}h)')
    print(f'{"="*60}')
    t0 = time.time()
    result_dist_w = distributed_window(
        data, window_h=WINDOW_H, with_network=WITH_NET,
        rho=RHO, tolerance=TOL, max_iterations=MAX_ITER)
    print(f'\nDistributed window: {time.time()-t0:.1f}s')
    if result_dist_w:
        print(f'Total cost: {result_dist_w["costs"].sum():.4f} EUR')
else:
    print('Centralized window failed — skipping distributed')
    result_dist_w = None

## 4. Window-by-window feasibility check

In [4]:
# Manually run each window for both solvers and record feasibility
print('Window-by-window feasibility check...')
print(f'{"Window":<8} {"t0:t1":<10} {"Centralized":<15} {"Distributed":<15}')
print('─' * 55)

soc_c = list(data['initial_soc'])
soc_d = list(data['initial_soc'])
ev_c  = list(data.get('ev_initial_soc', [0.0]*N))
ev_d  = list(data.get('ev_initial_soc', [0.0]*N))
T_c   = list(data.get('hp_T_init', [20.0]*N))
T_d   = list(data.get('hp_T_init', [20.0]*N))

window_log = []

for w in range(nw):
    t0_w, t1_w = w*Tw, (w+1)*Tw
    is_last = (w == nw-1)

    # Centralized window
    data_c = _slice_data(data, t0_w, t1_w, Tw)
    data_c.update({'initial_soc': list(soc_c),
                   'ev_initial_soc': list(ev_c),
                   'hp_T_init': list(T_c),
                   'cyclic_soc': is_last})
    try:
        rc = centralized(data_c, with_network=WITH_NET)
        c_status = f'✓ {rc["costs"].sum():.3f}'
        # Forward centralized state
        for i in range(N):
            if data['battery_capacity'][i] > 0:
                soc_c[i] = float(rc['soc'][i,-1])
            if data.get('ev_capacity',[0]*N)[i] > 0:
                ev_c[i] = float(rc['ev_soc'][i,-1])
            if data.get('hp_max_power',[0]*N)[i] > 0 and rc['temp'] is not None:
                T_c[i] = float(rc['temp'][i,-1])
    except Exception as e:
        c_status = f'✗ INFEASIBLE'
        rc = None

    # Distributed window — same initial state as centralized
    data_d = _slice_data(data, t0_w, t1_w, Tw)
    data_d.update({'initial_soc': list(soc_d),
                   'ev_initial_soc': list(ev_d),
                   'hp_T_init': list(T_d),
                   'cyclic_soc': is_last})
    try:
        rd = distributed(data_d, rho=RHO, tolerance=TOL,
                         max_iterations=MAX_ITER,
                         adaptive_rho=True, with_network=WITH_NET)
        d_status = f'✓ {rd["costs"].sum():.3f}  iter={rd["iterations"]}'
        # Forward distributed state
        for i in range(N):
            if data['battery_capacity'][i] > 0:
                soc_d[i] = float(rd['soc'][i,-1])
            if data.get('ev_capacity',[0]*N)[i] > 0:
                ev_d[i] = float(rd['ev_soc'][i,-1])
            if data.get('hp_max_power',[0]*N)[i] > 0 and rd['temp'] is not None:
                T_d[i] = float(rd['temp'][i,-1])
    except Exception as e:
        d_status = f'✗ INFEASIBLE'
        rd = None

    print(f'w={w+1:<5}  [{t0_w}:{t1_w}]  {c_status:<15}  {d_status}')
    window_log.append({
        'window': w+1, 't0': t0_w, 't1': t1_w,
        'cent_ok': rc is not None,
        'dist_ok': rd is not None,
        'soc_entering': list(soc_c),
        'ev_soc_entering': list(ev_c),
    })

Window-by-window feasibility check...
Window   t0:t1      Centralized     Distributed    
───────────────────────────────────────────────────────
w=1      [0:6]  ✓ 4.180          ✓ 4.179  iter=2
w=2      [6:12]  ✓ 6.187          ✓ 6.187  iter=2
w=3      [12:18]  ✓ 7.657          ✓ 7.657  iter=36
w=4      [18:24]  ✓ 4.776          ✓ 4.776  iter=101
w=5      [24:30]  ✓ 5.133          ✓ 5.133  iter=78
w=6      [30:36]  ✗ INFEASIBLE     ✗ INFEASIBLE
w=7      [36:42]  ✗ INFEASIBLE     ✗ INFEASIBLE
w=8      [42:48]  ✗ INFEASIBLE     ✗ INFEASIBLE


## 5. SOC trajectory comparison

In [5]:
if result_cent_w is not None and result_dist_w is not None:
    t_soc = np.arange(T+1) * dt
    t_day = np.arange(T) * dt
    C = ['#2F6FA3','#C65D3B','#2E6F52','#B5720A']

    fig, axes = plt.subplots(2, N, figsize=(4.5*N, 7), sharey='row')

    for i in range(N):
        # BESS SOC
        ax = axes[0, i]
        ax.plot(t_soc, result_full_c['soc'][i],
                color='k', lw=1.5, ls=':', label='Full-day cent.')
        if result_cent_w:
            ax.plot(t_soc, result_cent_w['soc'][i],
                    color=C[0], lw=2, label='Window cent.')
        if result_dist_w:
            ax.plot(t_soc, result_dist_w['soc'][i],
                    color=C[1], lw=2, ls='--', label='Window dist.')

        # Window boundaries
        for w in range(1, nw):
            ax.axvline(w * WINDOW_H, color='grey', lw=0.5, ls=':', alpha=0.5)

        cap = data['battery_capacity'][i]
        ax.axhline(data['soc_min_fraction']*cap, color='red',
                   lw=0.8, ls='--', alpha=0.6)
        ax.axhline(data['soc_max_fraction']*cap, color='red',
                   lw=0.8, ls='--', alpha=0.6)
        ax.set_title(f'P{i+1}: {names[i]}\nBESS SOC (kWh)', fontweight='bold', fontsize=8)
        ax.set_xlabel('Hour'); ax.set_ylabel('kWh')
        ax.set_xlim(0, T*dt)
        if i == 0: ax.legend(fontsize=7)

        # EV SOC
        ax = axes[1, i]
        ev_cap = data.get('ev_capacity',[0]*N)[i]
        if ev_cap > 0:
            ax.plot(t_soc, result_full_c['ev_soc'][i],
                    color='k', lw=1.5, ls=':', label='Full-day cent.')
            if result_cent_w:
                ax.plot(t_soc, result_cent_w['ev_soc'][i],
                        color=C[0], lw=2, label='Window cent.')
            if result_dist_w:
                ax.plot(t_soc, result_dist_w['ev_soc'][i],
                        color=C[1], lw=2, ls='--', label='Window dist.')
            cap_eff = ev_cap * data.get('ev_cap_factor',[1.]*N)[i]
            floor   = (1-data['ev_dod_max']) * cap_eff
            ax.axhline(floor, color='red', lw=0.8, ls='--', alpha=0.6,
                       label=f'DoD floor ({floor:.1f} kWh)')
            for w in range(1, nw):
                ax.axvline(w * WINDOW_H, color='grey', lw=0.5, ls=':', alpha=0.5)
        else:
            ax.text(0.5, 0.5, 'No EV', transform=ax.transAxes,
                    ha='center', va='center', color='grey')
        ax.set_title(f'EV SOC (kWh)', fontsize=8)
        ax.set_xlabel('Hour'); ax.set_ylabel('kWh')
        ax.set_xlim(0, T*dt)
        if i == 0: ax.legend(fontsize=6)

    plt.suptitle(
        f'SOC trajectories — {WINDOW_H}h windows\n'
        f'Dotted grey lines = window boundaries · Red dashed = SOC limits',
        fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('One or both windowed solves failed — cannot plot comparison')

One or both windowed solves failed — cannot plot comparison


## 6. Cost per window

In [6]:
# Rerun window-by-window with independent state to get per-window costs
# This uses centralized state forwarding for both — apples-to-apples comparison
print('Per-window cost breakdown (centralized state forwarding for both):')
print(f'{"Window":<8} {"Hours":<10}', end='')
for i in range(N): print(f' {"P"+str(i+1)+" cent":>10} {"P"+str(i+1)+" dist":>10}', end='')
print()
print('─' * (18 + 22*N))

soc_s = list(data['initial_soc'])
ev_s  = list(data.get('ev_initial_soc', [0.0]*N))
T_s   = list(data.get('hp_T_init', [20.0]*N))

for w in range(nw):
    t0_w, t1_w = w*Tw, (w+1)*Tw
    is_last = (w == nw-1)
    h0, h1 = t0_w*dt, t1_w*dt

    data_w = _slice_data(data, t0_w, t1_w, Tw)
    data_w.update({'initial_soc': list(soc_s),
                   'ev_initial_soc': list(ev_s),
                   'hp_T_init': list(T_s),
                   'cyclic_soc': is_last})

    print(f'w={w+1:<5}  {h0:.1f}–{h1:.1f}h', end='')

    try:
        rc = centralized(data_w, with_network=WITH_NET)
        for i in range(N): print(f' {rc["costs"][i]:>10.4f}', end='')
    except:
        for i in range(N): print(f' {"INFEAS":>10}', end='')
        rc = None

    try:
        rd = distributed(data_w, rho=RHO, tolerance=TOL,
                         max_iterations=MAX_ITER, adaptive_rho=True)
        for i in range(N): print(f' {rd["costs"][i]:>10.4f}', end='')
    except:
        for i in range(N): print(f' {"INFEAS":>10}', end='')
        rd = None
    print()

    # Forward state from centralized
    if rc is not None:
        for i in range(N):
            if data['battery_capacity'][i] > 0:
                soc_s[i] = float(rc['soc'][i,-1])
            if data.get('ev_capacity',[0]*N)[i] > 0:
                ev_s[i] = float(rc['ev_soc'][i,-1])
            if data.get('hp_max_power',[0]*N)[i] > 0 and rc['temp'] is not None:
                T_s[i] = float(rc['temp'][i,-1])

Per-window cost breakdown (centralized state forwarding for both):
Window   Hours         P1 cent    P1 dist    P2 cent    P2 dist    P3 cent    P3 dist    P4 cent    P4 dist
──────────────────────────────────────────────────────────────────────────────────────────────────────────
w=1      0.0–3.0h     0.1069     3.8142     0.1882     0.0704     0.1069     3.8141     0.1881     0.0703
w=2      3.0–6.0h     0.5702     4.7830     0.2453     0.5887     0.5702     4.7829     0.2452     0.5887
w=3      6.0–9.0h     0.2584     5.4447     0.1623     1.7917     0.2584     5.4447     0.1623     1.7917
w=4      9.0–12.0h    -0.4599     3.5797     0.1079     1.5484    -0.4599     3.5797     0.1079     1.5484
w=5      12.0–15.0h     0.2218     3.2607     0.2770     1.3731     0.1994     3.2720     0.2770     1.3844
w=6      15.0–18.0h     INFEAS     INFEAS     INFEAS     INFEAS     INFEAS     INFEAS     INFEAS     INFEAS
w=7      18.0–21.0h     INFEAS     INFEAS     INFEAS     INFEAS     INFEAS   

## 7. Vary window size
Sweep window sizes to find the smallest that stays feasible.

In [7]:
window_sizes = [1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0, 12.0]
# Filter to sizes that divide the day evenly
window_sizes = [w for w in window_sizes
                if T % int(round(w/dt)) == 0]
print(f'Valid window sizes for T={T}, dt={dt}h: {window_sizes}')

feasibility = []
for wh in window_sizes:
    print(f'\nWindow={wh}h ...', flush=True)
    rc = centralized_window(data, window_h=wh, with_network=WITH_NET)
    cent_ok = rc is not None
    dist_ok = False
    if cent_ok:
        try:
            rd = distributed_window(data, window_h=wh, with_network=WITH_NET,
                                    rho=RHO, tolerance=TOL,
                                    max_iterations=MAX_ITER)
            dist_ok = rd is not None
        except:
            dist_ok = False
    feasibility.append({'window_h': wh, 'cent_ok': cent_ok, 'dist_ok': dist_ok})
    print(f'  cent={"✓" if cent_ok else "✗"}  dist={"✓" if dist_ok else "✗"}')

df_feas = pd.DataFrame(feasibility).set_index('window_h')
display(df_feas)

Valid window sizes for T=48, dt=0.5h: [1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0, 12.0]

Window=1.0h ...
Centralized sliding window: 24 × 1.0h (2 steps)
  Window 1/24  t=[0:2)  (free terminal)
    soc_now:    [4.5, 6.0, 0.0, 3.0]
    ev_soc_now: [43.2, 28.8, 0.0, 0.0]
    T_now:      [20.0, 20.0, 20.0, 20.0]
    ✓ cost=0.0678 EUR
  Window 2/24  t=[2:4)  (free terminal)
    soc_now:    [1.5, 2.0, 0.0, 1.0]
    ev_soc_now: [42.926, 28.375, 0.0, 0.0]
    T_now:      [20.0, 20.0, 20.0, 20.0]
    ✓ cost=2.0896 EUR
  Window 3/24  t=[4:6)  (free terminal)
    soc_now:    [1.5, 2.0, 0.0, 1.0]
    ev_soc_now: [42.653, 27.95, 0.0, 0.0]
    T_now:      [20.0, 20.0, 20.0, 20.0]
    ✓ cost=2.0829 EUR
  Window 4/24  t=[6:8)  (free terminal)
    soc_now:    [1.5, 2.0, 0.0, 1.0]
    ev_soc_now: [42.379, 27.525, 0.0, 0.0]
    T_now:      [20.0, 20.0, 20.0, 20.0]
    ✓ cost=2.0301 EUR
  Window 5/24  t=[8:10)  (free terminal)
    soc_now:    [1.5, 2.0, 0.0, 1.0]
    ev_soc_now: [42.106, 27.1, 0.0, 0.0]
    T_now

,cent_ok,dist_ok
window_h,,
1.0,False,False
1.5,False,False
2.0,False,False
3.0,False,False
4.0,False,False
6.0,False,False
8.0,False,False
12.0,True,True
